In [1]:
import torch 
import torch.nn as nn
import torch.optim as optim 


import torchvision 
from torchvision.datasets import CIFAR10

In [2]:
# Datasets & DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = CIFAR10(root="./dataCNN", train=True, download=False, transform=transform)
testset = CIFAR10(root="./dataCNN", train=False, download=False, transform=transform)

In [3]:
trainloader=DataLoader(trainset,batch_size=64,shuffle=True)
testloader=DataLoader(testset,batch_size=64)

####BUIDING CNN

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
            
            nn.Conv2d (3,32, kernel_size=3 , padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d (32,64, kernel_size=3 , padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d (64,128, kernel_size=3 , padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)

        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layers(x)

        return x
        
        

In [5]:
model=CNN()

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [7]:
# epochs = 10

# for epoch in range(epochs):
#     epoch_training_loss = 0.0

#     for images, labels in trainloader:
#         optimizer.zero_grad()
        
#         output = model.forward(images) # FP
#         loss = criterion(output, labels) # loss fnx
#         loss.backward() # BP
#         optimizer.step() # update params

#         epoch_training_loss += loss.item()

#     print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

In [8]:
epochs = 10

for epoch in range(epochs):
    model.train()
    training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        
        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params

        training_loss += loss.item()

    epoch_train_loss = training_loss / len(trainloader) 


    model.eval()
    val_loss=0.0

    with torch.no_grad():
        for images,labels in testloader:
            output = model.forward(images) 
            loss = criterion(output, labels)
            # optimizer.step()

            val_loss+=loss.item()

        epoch_val_loss = val_loss / len(testloader)
            

    print(f"epoch={epoch+1}/{epochs} ==> training_loss={epoch_train_loss} & validation_loss={epoch_val_loss}")

epoch=1/10 ==> training_loss=1.3708822014539137 & validation_loss=1.0700539752935907
epoch=2/10 ==> training_loss=0.9389658199475549 & validation_loss=0.8732927900970362
epoch=3/10 ==> training_loss=0.7405441908732705 & validation_loss=0.7844822346025212
epoch=4/10 ==> training_loss=0.6041501207882182 & validation_loss=0.7576736297197403
epoch=5/10 ==> training_loss=0.4971875061883646 & validation_loss=0.7586843115129288
epoch=6/10 ==> training_loss=0.394700074020554 & validation_loss=0.7693064347573906
epoch=7/10 ==> training_loss=0.309388827646861 & validation_loss=0.8448058763507066
epoch=8/10 ==> training_loss=0.2297652891939482 & validation_loss=0.9151024778557432
epoch=9/10 ==> training_loss=0.18076346686486242 & validation_loss=1.0387031994048197
epoch=10/10 ==> training_loss=0.14186718655259484 & validation_loss=1.079354594277728


In [9]:
# Evaludate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        max_val , predicted  = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 75.25


In [11]:
# save the model

torch.save(model.state_dict(), "cnn_cifar10.pth")
print("model saved")

model saved
